## NMF Analysis

This notebook loads the processed motor dataset, normalizes features to a common scale, and prepares the matrix for NMF.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Config

df = pd.read_csv('processed_motor_data.csv')
print(f'Dataset shape: {df.shape}')
print('Columns:', list(df.columns))


In [ ]:
# Select features and normalize to [0, 1]
# Keep meta columns separate
meta_cols = ['time (s)', 'protein', 'DNA nM']
# Common measurement columns expected
feature_cols = [
    'Protein Concentration_nM', 'vorticity [1/s]_mean', 'divergence [1/s]_mean',
    'velocity magnitude [m/s]_mean', 'power [W]_mean', 'shear [1/s]_mean',
    'strain [1/s]_mean', 'correlation length [m]_mean', 'work [J]_mean',
    'Translation Rate [nM/s]'
]

# Keep only columns that exist
feature_cols = [c for c in feature_cols if c in df.columns]
use_cols = meta_cols + feature_cols
missing_meta = [c for c in meta_cols if c not in df.columns]
if missing_meta:
    print('Warning: missing meta columns:', missing_meta)

df_use = df[use_cols].copy()
print('Using columns:', use_cols)

# Drop rows where all features are NaN
feature_matrix = df_use[feature_cols]
non_all_nan_mask = ~feature_matrix.isna().all(axis=1)
df_use = df_use[non_all_nan_mask].reset_index(drop=True)

# Fit MinMaxScaler on features only
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(df_use[feature_cols].fillna(0))

# Put back into dataframe
df_scaled = df_use.copy()
for i, c in enumerate(feature_cols):
    df_scaled[c] = scaled[:, i]

print('Normalized feature ranges to [0, 1].')
df_scaled.head()


In [ ]:
# Build normalized time × condition matrices (no saving)
assert all(c in df_scaled.columns for c in ['time (s)', 'protein', 'DNA nM']), 'Missing meta columns in df_scaled'

df_scaled['condition'] = df_scaled['protein'].astype(str) + '-' + df_scaled['DNA nM'].astype(str) + 'nM'

feature_matrices = {}
for feature in feature_cols:
    print(f'Creating matrix for {feature}...')
    mat = df_scaled.pivot_table(
        index='time (s)',
        columns='condition',
        values=feature,
        aggfunc='mean'
    )
    # Interpolate only between first and last valid values per column
    mat_interp = mat.copy()
    for col in mat_interp.columns:
        s = mat_interp[col]
        first_valid = s.first_valid_index()
        last_valid = s.last_valid_index()
        if first_valid is not None and last_valid is not None:
            s_interp = s.loc[first_valid:last_valid].interpolate(method='linear', limit_direction='both')
            s_new = pd.Series(np.nan, index=s.index)
            s_new.loc[first_valid:last_valid] = s_interp
            mat_interp[col] = s_new
    feature_matrices[feature] = mat_interp

print('All feature matrices created (normalized).')
# Example preview
{feat: feature_matrices[feat].shape for feat in list(feature_matrices.keys())[:3]}


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize normalized matrices for selected/all features
features_to_show = feature_cols  # or set a subset like: ['velocity magnitude [m/s]_mean']

for feature in features_to_show:
    if feature not in feature_matrices:
        print(f"Skipping {feature}: not in feature_matrices")
        continue
    mat = feature_matrices[feature]
    mat_vis = mat.fillna(0)

    plt.figure(figsize=(16, 10))
    ax = sns.heatmap(
        mat_vis.T,
        cmap='viridis',
        cbar_kws={'label': f'{feature} (normalized [0,1])'},
        xticklabels=120,
        yticklabels=True,
    )
    # Convert x tick labels (seconds) to hours
    xticks = ax.get_xticks()
    if len(xticks) > 0:
        try:
            xtick_vals = mat_vis.index.values[xticks.astype(int)]
            xtick_hours = xtick_vals / 3600
            ax.set_xticklabels([f"{x:.1f}" for x in xtick_hours])
        except Exception:
            pass

    plt.title(f'{feature} over Time by Condition (Normalized)', fontsize=14, fontweight='bold')
    plt.xlabel('Time (h)')
    plt.ylabel('Condition (protein-DNA)')
    plt.tight_layout()
    plt.show()
